# EDA & Feature Engineering

Khám phá dữ liệu sau preprocessing, phân tích phân phối, outlier, tương quan, đa cộng tuyến, và trực quan hóa các pattern quan trọng.

| Phân tích | Mục đích |
|-----------|----------|
| Distribution + Boxplot | Nhận diện skewness, outliers, so sánh delay vs non-delay |
| Delay rate by category | Tìm category có tỷ lệ delay cao → insight business |
| Interaction analysis | Phát hiện tương tác giữa 2 biến phân loại |
| Spearman correlation | Đo lường tương quan phi tuyến (robust với outliers) |
| VIF | Phát hiện đa cộng tuyến giữa các biến numeric |
| Feature engineering demo | Minh họa time features và binning |


## 2.1 Setup & Load Data

Load config và dữ liệu đã qua preprocessing. Dữ liệu được chia riêng cho 2 kỳ để so sánh pattern giữa April–June và July–September.


In [ ]:
"""Configuration constants for DS108 Lab 4 pipeline."""
import os

# Paths
DATA_DIR = "Data"
RESULTS_DIR = "results"
EDA_DIR = os.path.join(RESULTS_DIR, "eda")
EXP_DIR = os.path.join(RESULTS_DIR, "experiments")
MODEL_DIR = os.path.join(RESULTS_DIR, "models")
PLOT_DIR = os.path.join(RESULTS_DIR, "plots")

# Raw files
DELAY_46 = os.path.join(DATA_DIR, "delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_46 = os.path.join(DATA_DIR, "not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv")
DELAY_79 = os.path.join(DATA_DIR, "delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")
NOT_DELAY_79 = os.path.join(DATA_DIR, "not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv")

# Experiment labels
PERIOD_A = "7_9"   # future
PERIOD_B = "4_6"   # past

# Random seed for reproducibility
RANDOM_STATE = 42

# Train/Val/Test split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Stratified K-Fold
N_SPLITS = 5

# Incremental learning ratios for alpha_4
INCREMENTAL_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]

# Model light hyperparameters (tuned via CV inside training)
LGBM_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
    "is_unbalance": True,
}

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "n_estimators": 1000,
    "random_state": RANDOM_STATE,
}

CAT_PARAMS = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "auto_class_weights": "Balanced",
}

KNN_PARAMS = {
    "n_neighbors": 5,
}

        import pandas as pd
        import numpy as np
        import matplotlib.pyplot as plt
        import seaborn as sns
        import os
        sns.set_style("whitegrid")
        plt.rcParams["figure.figsize"] = (10, 6)

        def load_period(delay_path, not_delay_path, label):
            df_delay = pd.read_csv(delay_path, low_memory=False)
            df_not = pd.read_csv(not_delay_path, low_memory=False)
            df_delay["label"] = label
            df_not["label"] = 1 - label
            return pd.concat([df_delay, df_not], ignore_index=True)

        df_46 = load_period(config.DELAY_46, config.NOT_DELAY_46, 1)
        df_79 = load_period(config.DELAY_79, config.NOT_DELAY_79, 1)
        print(f"4-6: {df_46.shape}, 7-9: {df_79.shape}")

### Tổng quan dữ liệu

- Kỳ 4–6: ~2.9M rows, tỷ lệ delay ~2.5%
- Kỳ 7–9: ~6.3M rows, tỷ lệ delay ~2.3% (giảm nhẹ so với kỳ trước)
- Tỷ lệ lớp cực kỳ mất cân bằng (~1:40) → cần xử lý ở bước Modeling.


## 2.2 Distribution & Boxplot Analysis

Vẽ histogram kèm KDE và boxplot song song cho từng biến numeric.
Giúp nhận diện skewness, outliers, và phân phối bất đối xứng.

> Hầu hết biến numeric có nhiều outliers và phân phối lệch phải (right-skewed). Ta dùng Winsorization thay vì xóa outliers để giữ nguyên kích thước dataset.


In [ ]:
def plot_distribution_boxplot(df, num_cols=None, period="", save_path=None):
    if num_cols is None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if "label" in num_cols:
            num_cols.remove("label")
    n = len(num_cols)
    if n == 0:
        return
    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(12, 4 * n))
    if n == 1:
        axes = np.array([axes])
    sage_color = "#9DC183"
    for i, col in enumerate(num_cols):
        sns.histplot(df[col], kde=True, bins=30, color=sage_color, ax=axes[i, 0])
        axes[i, 0].set_title(f"Distribution of {col}")
        axes[i, 0].set_xlabel(col)
        axes[i, 0].set_ylabel("Count")
        sns.boxplot(y=df[col], color=sage_color, ax=axes[i, 1])
        axes[i, 1].set_title(f"Boxplot of {col}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# Demo on 4-6 numeric columns (first 5 for brevity)
num_cols_46 = ['Sales order line number', 'SO QTY', 'ALLOCATION QTY',
               'SUPPLIER INV AMOUNT', 'PURCHASE AMOUNT']
plot_distribution_boxplot(df_46, num_cols=num_cols_46, period="4-6")

### Nhận xét từ phân phối

- `SUPPLIER INV AMOUNT` và `WEIGHT PER PIECE`: phân phối lệch phải mạnh, có nhiều outliers nhưng cũng có sự khác biệt rõ rệt giữa 2 nhãn → **strong predictors**.
- `ALLOCATION QTY` và `ACTUAL_SHIP_DAYS`: sự khác biệt ít hơn → **moderate predictors**.
- `Sales order line number` và `PACK QTY`: phân phối gần như giống nhau giữa 2 nhãn → **weak predictors**, có thể cân nhắc loại bỏ.


## 2.3 Boxplot by Label

So sánh phân phối numeric giữa nhóm delay (`DELAY_FLG=1`) và không delay (`DELAY_FLG=0`).

Các biến có sự khác biệt rõ rệt về median/spread giữa 2 nhóm thường có predictive power cao hơn.


In [ ]:
def plot_boxplot_by_label(df, num_cols=None, period="", save_path=None):
    if num_cols is None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if "label" in num_cols:
            num_cols.remove("label")
    n = len(num_cols)
    fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(8, 4 * n))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, num_cols):
        sns.boxplot(x="label", y=col, data=df, ax=ax, color="seagreen")
        ax.set_title(f"{col} by label")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_boxplot_by_label(df_46, num_cols=num_cols_46[:3])

### Lưu ý về cột ACTUAL_SHIP_DAYS

Cột `ACTUAL_SHIP_DAYS` ghi nhận sau khi đã giao hàng (post-delivery), không phải thông tin có sẵn lúc dự đoán. Nếu dùng cột này trong model sẽ gây **data leakage** — model "biết" kết quả trước khi dự đoán.

→ Cột này bị loại bỏ trong quá trình preprocessing.


## 2.4 Delay Rate by Category

Phân tích tỷ lệ delay theo các biến phân loại quan trọng:
- `SUPPLIER_DIV`, `Ship Mode`, `Order_Size_Group`, `Value_Group`

Sử dụng colormap `RdYlGn_r` để thể hiện mức độ delay (đỏ = cao, xanh = thấp).

**Insights từ notebook tham khảo:**
- **Supplier Division**: Div 4 có delay rate cao nhất ở kỳ 4–6, nhưng cải thiện đáng kể ở kỳ 7–9.
- **Ship Mode**: Mode X và O có delay rate cực cao (50–66%). Mode X chỉ xuất hiện ở Division 3.
- **Supplier Category**: Category 6 có delay rate ~20%, gấp đôi các category còn lại.


In [ ]:
def plot_delay_rate_by_category(df, cat_cols=None, top_n=5, save_path=None):
    if cat_cols is None:
        cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
        if "label" in cat_cols:
            cat_cols.remove("label")
    cat_cols = [c for c in cat_cols if df[c].nunique() < 100][:top_n]
    if not cat_cols:
        return
    n = len(cat_cols)
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten() if n > 1 else [axes]
    for idx, col in enumerate(cat_cols):
        rate = df.groupby(col)["label"].mean().sort_values(ascending=False)
        bars = axes[idx].bar(range(len(rate)), rate.values * 100,
                             color=plt.cm.RdYlGn_r(rate.values))
        axes[idx].set_xticks(range(len(rate)))
        axes[idx].set_xticklabels(rate.index, rotation=45, ha="right")
        axes[idx].set_ylabel("Delay Rate (%)")
        axes[idx].set_title(f"Delay Rate by {col}")
        axes[idx].grid(axis="y", alpha=0.3)
    for idx in range(len(cat_cols), len(axes)):
        axes[idx].set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# Create binned features for demo
bins = pd.qcut(df_46["SO QTY"], q=3, duplicates="drop", retbins=True)[1]
labels = ["Small", "Medium", "Large"][:len(bins)-1]
df_46["Order_Size_Group"] = pd.qcut(df_46["SO QTY"], q=3, duplicates="drop", labels=labels).astype(str)
bins = pd.qcut(df_46["SUPPLIER INV AMOUNT"], q=3, duplicates="drop", retbins=True)[1]
labels = ["Low", "Medium", "High"][:len(bins)-1]
df_46["Value_Group"] = pd.qcut(df_46["SUPPLIER INV AMOUNT"], q=3, duplicates="drop", labels=labels).astype(str)

plot_delay_rate_by_category(df_46, cat_cols=["SUPPLIER_DIV", "Ship Mode",
                                              "Order_Size_Group", "Value_Group"])

## 2.5 Stacked Bar – On-time vs Delayed

Biểu đồ stacked bar cho từng category, thể hiện tỷ lệ on-time và delayed.
Giúp trực quan hóa mức độ imbalance trong từng nhóm category.


In [ ]:
def plot_stacked_delay_distribution(df, cat_col, save_path=None):
    crosstab = pd.crosstab(df[cat_col], df["label"], normalize="index") * 100
    fig, ax = plt.subplots(figsize=(10, 5))
    crosstab.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#e74c3c"], width=0.7)
    ax.set_xlabel(cat_col)
    ax.set_ylabel("Percentage (%)")
    ax.set_title(f"On-time vs Delayed by {cat_col}")
    ax.legend(["On-time", "Delayed"], loc="best")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_stacked_delay_distribution(df_46, "Ship Mode")

## 2.6 Interaction Analysis

Phân tích tương tác giữa các biến phân loại:
- **Line plot**: delay rate theo từng combination của 2 biến
- **Heatmap**: ma trận delay rate giữa 2 biến

Mục đích: phát hiện các cặp biến có tương tác mạnh (ví dụ: Ship Mode X + Division 3 có delay rate 100% ở kỳ 7–9).


In [ ]:
def plot_interactions(df, pairs=None, save_path=None):
    if pairs is None:
        pairs = []
        if "Ship Mode" in df.columns and "Order_Size_Group" in df.columns:
            pairs.append(("Ship Mode", "Order_Size_Group"))
        if "SUPPLIER_DIV" in df.columns and "Ship Mode" in df.columns:
            pairs.append(("SUPPLIER_DIV", "Ship Mode"))
    if not pairs:
        print("No interaction pairs found.")
        return
    n = len(pairs)
    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(16, 6 * n))
    if n == 1:
        axes = np.array([axes])
    for i, (col1, col2) in enumerate(pairs):
        interaction = df.groupby([col1, col2])["label"].mean().unstack()
        for col in interaction.columns:
            axes[i, 0].plot(interaction.index, interaction[col] * 100,
                           marker="o", linewidth=2, markersize=8, label=col)
        axes[i, 0].set_xlabel(col1)
        axes[i, 0].set_ylabel("Delay Rate (%)")
        axes[i, 0].set_title(f"Interaction: {col1} × {col2}")
        axes[i, 0].legend(title=col2)
        axes[i, 0].grid(alpha=0.3)
        axes[i, 0].tick_params(axis="x", rotation=45)

        heatmap_data = df.groupby([col1, col2])["label"].mean().unstack() * 100
        sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd",
                    ax=axes[i, 1], cbar_kws={"label": "Delay Rate (%)"})
        axes[i, 1].set_xlabel(col2)
        axes[i, 1].set_ylabel(col1)
        axes[i, 1].set_title(f"Heatmap: {col1} × {col2}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_interactions(df_46)

## 2.7 Spearman Correlation

Dùng **Spearman** thay vì Pearson vì phân phối không chuẩn và có nhiều outliers.
Spearman dựa trên **rank** nên robust hơn với outliers.

> Các biến có correlation cao với nhau có thể gây đa cộng tuyến → cần kiểm tra VIF ở bước sau.


In [ ]:
def plot_spearman_heatmap(df, period="", save_path=None):
    numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")
    if numeric_df.shape[1] < 2:
        print("Not enough numeric columns.")
        return
    corr = numeric_df.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Spearman Correlation Matrix ({period})", fontsize=16, fontweight="bold", pad=20)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_spearman_heatmap(df_46, period="4-6")

## 2.8 VIF – Variance Inflation Factor

VIF đo lường mức độ đa cộng tuyến (multicollinearity) giữa các biến numeric:
- **VIF < 5**: không có đa cộng tuyến
- **VIF 5–10**: đa cộng tuyến trung bình
- **VIF > 10**: đa cộng tuyến nghiêm trọng → nên loại bỏ hoặc kết hợp biến


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def compute_vif(df):
    numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")
    numeric_df = numeric_df.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    X = add_constant(numeric_df)
    vif_results = []
    for i, col in enumerate(X.columns):
        if col == "const":
            continue
        try:
            vif_value = variance_inflation_factor(X.values, i)
            vif_results.append({"Feature": col, "VIF": vif_value})
        except Exception:
            vif_results.append({"Feature": col, "VIF": np.nan})
    return pd.DataFrame(vif_results).sort_values("VIF", ascending=False)

def plot_vif(vif_df, save_path=None):
    if vif_df.empty:
        return
    fig, ax = plt.subplots(figsize=(12, max(6, len(vif_df) * 0.4)))
    colors = ["#e74c3c" if x > 10 else "#f39c12" if x > 5 else "#2ecc71"
              for x in vif_df["VIF"].fillna(0)]
    bars = ax.barh(range(len(vif_df)), vif_df["VIF"], color=colors, alpha=0.7)
    ax.set_yticks(range(len(vif_df)))
    ax.set_yticklabels(vif_df["Feature"])
    ax.set_xlabel("VIF Value", fontsize=12, fontweight="bold")
    ax.set_title("Variance Inflation Factor (VIF)", fontsize=14, fontweight="bold")
    ax.axvline(x=5, color="orange", linestyle="--", linewidth=2, label="VIF = 5")
    ax.axvline(x=10, color="red", linestyle="--", linewidth=2, label="VIF = 10")
    ax.legend()
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, vif_df["VIF"]):
        if not np.isnan(val):
            ax.text(val, bar.get_y() + bar.get_height() / 2,
                    f"{val:.2f}", ha="left", va="center", fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

vif_df = compute_vif(df_46)
print(vif_df)
plot_vif(vif_df)

### Statistical Testing

Mặc dù không trực tiếp sử dụng trong pipeline hiện tại, các kiểm định thống kê (Chi-square cho categorical, Mann-Whitney U cho numeric) có thể giúp xác nhận sự khác biệt có ý nghĩa thống kê giữa nhóm delay và non-delay.


## 2.9 Time-based Feature Engineering Demo

Minh họa cách tạo các đặc trưng thời gian từ `SO_TIME`, Order date, và `VSD`.

`SO_TIME` ở định dạng HHMMSS (vd: 120324 = 12:03:24). Để parse đúng:
1. Chuyển về string với zero-padding 6 chữ số
2. Dùng `pd.to_datetime` với format `%H%M%S`
3. Extract giờ, sau đó phân loại Morning/Afternoon/Evening/Night


In [ ]:
df_time = df_46.copy()
df_time["Order date"] = pd.to_datetime(df_time["Order date"], errors="coerce")
df_time["VSD"] = pd.to_datetime(df_time["VSD"], errors="coerce")

# time_period
df_time["SO_TIME_str"] = df_time["SO_TIME"].astype(str).str.zfill(6)
hour = pd.to_datetime(df_time["SO_TIME_str"], format="%H%M%S", errors="coerce").dt.hour
df_time["time_period"] = hour.apply(
    lambda h: "Morning" if 5 <= h < 12 else
              "Afternoon" if 12 <= h < 17 else
              "Evening" if 17 <= h < 22 else
              "Night" if pd.notna(h) else "__MISSING__"
)

# IS_WEEKEND, MONTH_PHASE
df_time["IS_WEEKEND"] = df_time["SO_DAY_OF_WEEK"].isin([5, 6]).astype(int)
df_time["MONTH_PHASE"] = pd.cut(df_time["SO_DAY_OF_MONTH"], bins=[0, 7, 14, 31],
                                 labels=["early", "mid", "late"]).astype(str)

# Order/VSD month and phases
df_time["Order_month"] = df_time["Order date"].dt.month
df_time["VSD_month"] = df_time["VSD"].dt.month
df_time["VSD_IS_WEEKEND"] = df_time["VSD"].dt.dayofweek.isin([5, 6]).astype(int)
df_time["VSD_MONTH_PHASE"] = pd.cut(df_time["VSD"].dt.day, bins=[0, 7, 14, 31],
                                     labels=["early", "mid", "late"]).astype(str)

# Expected delivery days
df_time["Expected_delivery_days"] = (df_time["VSD"] - df_time["Order date"]).dt.days

# Plot delay rate by time features
time_cols = ["time_period", "IS_WEEKEND", "MONTH_PHASE", "Order_month",
             "VSD_month", "VSD_IS_WEEKEND", "VSD_MONTH_PHASE"]
n = len(time_cols)
n_cols = 2
n_rows = int(np.ceil(n / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

from matplotlib import cm
from matplotlib.colors import ListedColormap
custom_greens = ListedColormap(cm.Greens(np.linspace(0.4, 1.0, 256)))

for idx, col in enumerate(time_cols):
    if df_time[col].nunique() < 100:
        rate = df_time.groupby(col)["label"].mean()
        norm = plt.Normalize(0.01, rate.max())
        colors = [custom_greens(norm(val)) for val in rate.values]
        sns.barplot(x=rate.index, y=rate.values, palette=colors, ax=axes[idx])
        axes[idx].set_title(f"Delay rate by {col}")
        axes[idx].set_ylabel("Delay rate")
        axes[idx].tick_params(axis="x", rotation=45)

for idx in range(len(time_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 2.10 Export

Lưu dữ liệu đã xử lý và các artifacts EDA (correlation plots, summary CSV) cho bước Modeling.


In [ ]:
# Simple alignment + basic clean (without full preprocessing)
common = list(set(df_46.columns) & set(df_79.columns))
df_46_aligned = df_46[common].copy()
df_79_aligned = df_79[common].copy()

os.makedirs("processed", exist_ok=True)
df_46_aligned.to_csv("processed/tidy_4_6.csv", index=False)
df_79_aligned.to_csv("processed/tidy_7_9.csv", index=False)
print("Exported tidy data to processed/")